In [24]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

golden = pd.read_csv("../data/golden_set.csv")
pairs = pd.read_csv("../results/reply_pairs.csv")

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Keep only rows that have an escalation label
esc_data = golden.dropna(subset=["escalate"]).copy()

X = esc_data["text"]
y = esc_data["escalate"]

esc_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=3000
)

X_vec = esc_vectorizer.fit_transform(X)

esc_model = LogisticRegression(max_iter=1000)
esc_model.fit(X_vec, y)

print("Escalation model trained!")
print("Training samples:", len(esc_data))

Escalation model trained!
Training samples: 52


In [26]:
print("Training samples:", len(esc_data))
print(esc_data["escalate"].value_counts())

Training samples: 52
escalate
No     44
Yes     8
Name: count, dtype: int64


In [27]:
import joblib

joblib.dump(esc_model, "../results/escalation_model.pkl")
joblib.dump(esc_vectorizer, "../results/escalation_vectorizer.pkl")

print("Escalation model saved!")

Escalation model saved!


In [28]:
import joblib

intent_model = joblib.load("../results/intent_model.pkl")
intent_vectorizer = joblib.load("../results/vectorizer.pkl")

print("Intent model loaded successfully!")

Intent model loaded successfully!


In [29]:
reply_vectorizer = joblib.load("../results/reply_vectorizer.pkl")
reply_matrix = joblib.load("../results/reply_matrix.pkl")

print("Reply engine loaded!")

Reply engine loaded!


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_reply(message):

    vec = reply_vectorizer.transform([message])

    sim = cosine_similarity(vec, reply_matrix)

    idx = sim.argmax()

    return pairs.iloc[idx]["text_support"]

In [32]:
esc_model = joblib.load("../results/escalation_model.pkl")
esc_vectorizer = joblib.load("../results/escalation_vectorizer.pkl")

print("Escalation model loaded!")

Escalation model loaded!


In [33]:
def predict_support_agent(message):

    # Intent prediction
    intent = intent_model.predict(
        intent_vectorizer.transform([message])
    )[0]

    # Escalation prediction
    escalate = esc_model.predict(
        esc_vectorizer.transform([message])
    )[0]

    # Reply retrieval
    reply = retrieve_reply(message)

    return {
        "intent": intent,
        "escalate": escalate,
        "reply": reply
    }

In [34]:
tweet = "I was charged twice for Premium"

result = predict_support_agent(tweet)

print(result)

{'intent': 'General Inquiry', 'escalate': 'No', 'reply': "@671882 Hey Chino, that doesn't sound good! Can you DM us your account's email address? We'll take a look backstage /JN https://t.co/ldFdZRiNAt"}


In [35]:
predict_support_agent(
    "My music keeps skipping after the iOS update"
)

{'intent': 'Playback Issues',
 'escalate': 'No',
 'reply': "@526990 Hmm. Could you see if this is happening across all your devices? We'll get to the bottom of this /LP"}

In [36]:
predict_support_agent(
    "I can't log into my account"
)

{'intent': 'General Inquiry',
 'escalate': 'No',
 'reply': "@214781 Hey Andy! Can you tell us more about what's happening? We'll see what we can suggest /RH"}

In [37]:
print("AI Support Agent completed!")

AI Support Agent completed!
